*This is the first post in our new **Ask LatinCy** column.*

In [ ]:
from collections import Counter, defaultdict

import pandas as pd
import spacy
import tabulate
from tqdm import tqdm

In [ ]:
from latincyreaders import TesseraeReader, AnnotationLevel

T = TesseraeReader(annotation_level=AnnotationLevel.BASIC)

fileids = T.fileids()
print(f"Total texts in Tesserae corpus: {len(fileids)}")
print(f"\nFirst 10 texts:")
for f in fileids[:10]:
    print(f"  {f}")
print(f"  ...")

## Model accuracy

In [ ]:
# Load model metadata to report accuracy scores
import json
from pathlib import Path

nlp = spacy.load("la_core_web_lg")
meta = nlp.meta
perf = meta["performance"]

print(f"Model: la_core_web_lg v{meta['version']}")
print(f"spaCy version: {meta['spacy_version']}")
print()

# Key accuracy metrics for this post's frequency lists
accuracy_data = [
    ("POS tagging", f"{perf['pos_acc']:.2%}"),
    ("Lemmatization", f"{perf['lemma_acc']:.2%}"),
    ("Morphological features", f"{perf['morph_acc']:.2%}"),
    ("Fine-grained tags", f"{perf['tag_acc']:.2%}"),
]

print(tabulate.tabulate(
    accuracy_data,
    headers=["Component", "Accuracy"],
    tablefmt="simple"
))

In [ ]:
# Process the full corpus in a single pass, collecting:
# - form counts (token.text)
# - lemma+POS counts (token.lemma_, token.pos_)
# - per-form lemma+POS distributions (for disambiguation)
# - total token count

form_counter = Counter()
lemma_pos_counter = Counter()
form_to_lemma_pos = defaultdict(Counter)
total_tokens = 0

for doc in tqdm(T.docs(fileids=fileids), total=len(fileids), desc="Processing corpus"):
    for token in doc:
        if not token.is_alpha:
            continue
        form = token.text.lower()
        lemma_pos = (token.lemma_, token.pos_)
        form_counter[form] += 1
        lemma_pos_counter[lemma_pos] += 1
        form_to_lemma_pos[form][lemma_pos] += 1
        total_tokens += 1

print(f"\nTotal alphabetic tokens: {total_tokens:,}")
print(f"Unique forms: {len(form_counter):,}")
print(f"Unique lemma+POS pairs: {len(lemma_pos_counter):,}")

## Top forms

In [ ]:
top_n = 50

top_forms = form_counter.most_common(top_n)

form_data = []
cumulative = 0
for rank, (form, count) in enumerate(top_forms, 1):
    pct = count / total_tokens * 100
    cumulative += pct
    form_data.append((rank, form, f"{count:,}", f"{pct:.2f}%", f"{cumulative:.2f}%"))

print(tabulate.tabulate(
    form_data,
    headers=["Rank", "Form", "Count", "% of total", "Cumulative %"],
    tablefmt="simple"
))

## Top lemma+POS pairs

In [ ]:
top_lemma_pos = lemma_pos_counter.most_common(top_n)

lemma_pos_data = []
cumulative = 0
for rank, ((lemma, pos), count) in enumerate(top_lemma_pos, 1):
    pct = count / total_tokens * 100
    cumulative += pct
    explanation = spacy.explain(pos)
    lemma_pos_data.append((rank, lemma, pos, explanation, f"{count:,}", f"{pct:.2f}%", f"{cumulative:.2f}%"))

print(tabulate.tabulate(
    lemma_pos_data,
    headers=["Rank", "Lemma", "POS", "POS (expanded)", "Count", "% of total", "Cumulative %"],
    tablefmt="simple"
))

## Disambiguation: the case of *cum*

In [ ]:
# Show all lemma+POS entries for a given form to demonstrate disambiguation

def show_form_disambiguation(form, form_counter, form_to_lemma_pos):
    """Show how a single form maps to different lemma+POS pairs."""
    total = form_counter[form]
    pairs = form_to_lemma_pos[form].most_common()
    data = []
    for (lemma, pos), count in pairs:
        pct = count / total * 100
        data.append((form, lemma, pos, spacy.explain(pos), f"{count:,}", f"{pct:.1f}%"))
    print(tabulate.tabulate(
        data,
        headers=["Form", "Lemma", "POS", "POS (expanded)", "Count", "% of form"],
        tablefmt="simple"
    ))
    print()

In [ ]:
ambiguous_forms = ["cum", "quod", "ut", "qui", "que", "ne"]

In [ ]:
for form in ambiguous_forms:
    show_form_disambiguation(form, form_counter, form_to_lemma_pos)

## Export frequency lists

In [ ]:
# Export top 10,000 forms

top_10k_forms = form_counter.most_common(10_000)

forms_df = pd.DataFrame(top_10k_forms, columns=["form", "count"])
forms_df.index = range(1, len(forms_df) + 1)
forms_df.index.name = "rank"
forms_df["pct"] = (forms_df["count"] / total_tokens * 100).round(4)
forms_df["cumulative_pct"] = forms_df["pct"].cumsum().round(4)

forms_df.to_csv("tesserae_top_10k_forms.csv")
print(f"Exported {len(forms_df)} forms to tesserae_top_10k_forms.csv")
forms_df.head(10)

In [ ]:
# Export top 10,000 lemma+POS pairs

top_10k_lemma_pos = lemma_pos_counter.most_common(10_000)

lemma_pos_df = pd.DataFrame(
    [(lemma, pos, count) for (lemma, pos), count in top_10k_lemma_pos],
    columns=["lemma", "pos", "count"]
)
lemma_pos_df.index = range(1, len(lemma_pos_df) + 1)
lemma_pos_df.index.name = "rank"
lemma_pos_df["pct"] = (lemma_pos_df["count"] / total_tokens * 100).round(4)
lemma_pos_df["cumulative_pct"] = lemma_pos_df["pct"].cumsum().round(4)

lemma_pos_df.to_csv("tesserae_top_10k_lemma_pos.csv")
print(f"Exported {len(lemma_pos_df)} lemma+POS pairs to tesserae_top_10k_lemma_pos.csv")
lemma_pos_df.head(10)

## Lemma-only frequencies (aggregated across POS)

In [ ]:
# Aggregate lemma counts across all POS tags

lemma_counter = Counter()
for (lemma, pos), count in lemma_pos_counter.items():
    lemma_counter[lemma] += count

top_lemmas = lemma_counter.most_common(top_n)

# For each lemma, show all POS tags it appears with
lemma_to_pos = defaultdict(set)
for (lemma, pos), count in lemma_pos_counter.items():
    lemma_to_pos[lemma].add(pos)

lemma_data = []
cumulative = 0
for rank, (lemma, count) in enumerate(top_lemmas, 1):
    pct = count / total_tokens * 100
    cumulative += pct
    pos_tags = ", ".join(sorted(lemma_to_pos[lemma]))
    lemma_data.append((rank, lemma, pos_tags, f"{count:,}", f"{pct:.2f}%", f"{cumulative:.2f}%"))

print(tabulate.tabulate(
    lemma_data,
    headers=["Rank", "Lemma", "POS tags", "Count", "% of total", "Cumulative %"],
    tablefmt="simple"
))